# Data Creation and Collection Lab 2: Working with Models

## Introduction

A very important aspect of supervised and semi-supervised machine learning is the quality of the labels produced by human labelers. Unfortunately, humans are not perfect and in some cases may even maliciously label things incorrectly. In the previous lab, you gained some hands-on experience with an example of such messy data, and how to "clean" it. In this lab, you will take the next step and explore using the cleaned data to train a model. You will then tackle evaluating that model. At the end, you will intentionally mess-ify your data and re-run your models to see the effects of bad data on a model's performance.

Throughout, we have provided additional resources in the form of blogs to help familiarize you with some of the concepts used in this lab. Additionally, we have provided a number of code snippets you can use during this assignment. Feel free to modify them or replace them if you wish.

## Dataset
The dataset you will be using for the lab is the [Adult Income dataset](https://archive.ics.uci.edu/ml/datasets/Adult). Be sure to download the data following that link. This dataset was created by Ronny Kohavi and Barry Becker and was used to predict whether a person's income is more/less than 50k USD based on census data [[1]](http://robotics.stanford.edu/~ronnyk/nbtree.pdf).

## Submission for Verified Learners

There will be three distinct values you will submit on the course page to complete this lab. Each value is marked as being required for submission in its respective section. These occur in the **Data Visualization**, **Putting It All Together**, and **Label Perturbation** Sections.

### Data Preprocessing
This dataset is intended to be used to train and evaluate models for predicting if an individual has income greater than 50k. There exists a column in this dataset that tells you the answer (column index 14), this is called the target variable. To begin using the data to train a model to make predictions, you must first start by loading and preprocessing the data. Remove NaN values, convert strings to categorical variables and encode the target variable (the string <=50K, >50K in column index 14). You can read more on categorical data in Pandas [here](https://pandas.pydata.org/pandas-docs/stable/user_guide/categorical.html), and more on encoding values [here](https://pbpython.com/categorical-encoding.html).

**NOTE**: When doing the target variable encoding, ensure that the `>50k` is the positive label.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# This can be used to load the dataset
data = pd.read_csv(Path().cwd().joinpath("adult_all.csv"), header=None, na_values='?')
data.head()

In [ ]:
# Code

### Data visualization
A simple strategy to identify which features are good indicators of a particular target variable is to calculate the correlation between the different features and the target variable. Visualizing the correlations in a heatmap can make this process more interpretable for you. A good example of how to do calculate and visualize these correlations can be found [here](https://towardsdatascience.com/better-heatmaps-and-correlation-matrix-plots-in-python-41445d0f2bec). **Hint**: If you follow this strategy to create a heatmap, be sure to include an additional argument (`annot=True`) in the call to heatmap to add the numerical values to the visualization.

Identify the features that have the highest, and the lowest correlation with the target variable. Record these features, as you will submit these at the end of the lab.

In [ ]:
# Code

### Data classification

In this section, you will be implementing and evaluating an [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) classifier.

#### Preprocessing
We gave some simple pre-processing steps earlier, to give you additional practice working with data. You can automate this process using something called a Pipeline. When designing a pipeline, you need to think about how you are going to encode any categorical variables that may be present, and whether or not you want to use all of the features available in the data. For this section, we will be using a process called One Hot Encoding. For an example of what this looks like, refer back to Approach #3 in [this blog](https://pbpython.com/categorical-encoding.html).

For further information on Pipelines, read more about them [here](https://machinelearningmastery.com/columntransformer-for-numerical-and-categorical-data/) and [here](https://medium.com/vickdata/a-simple-guide-to-scikit-learn-pipelines-4ac0d974bdcf).

#### Separating Features and Targets

An important step in preparing a machine learning model is separating the features from the target. Conventionally, the features are defined as `X`, and the target as `y`. You can see this convention applied in our provided boiler plate. However, it is not always just as simple as separating the two and feeding it into a model. You also need to consider which data the model is meant to learn from, i.e., the *train* data, and which data is meant to evaluate the model, i.e., the *test* data. You can define train and test manually using the [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html#sklearn.model_selection.train_test_split) method, or you can keep it all together and use a process called [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html). For this lab, we ask you to use the KFold approach with the number of folds set to 10.

#### Evaluation
In order to understand how well your model is predicting your target variable, it must be evaluated. There are many different metrics available for this purpose, some you have been introduced to in the lecture videos. Additionally, you can find more information on different metrics on [Scikit-Learn's Documentation](https://scikit-learn.org/stable/modules/model_evaluation.html). Scikit-Learn is a powerful Python machine learning library. It is good to explore the different metrics and what they measure. In this lab, we will be using [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html#sklearn.model_selection.cross_val_score) set to simple `accuracy`, for ease of grading. You should get an accuracy score for each fold in KFold. Submit the average of these accuracies, out to 3 decimals.

#### Putting It All Together

Below, we have provided some boilerplate code. It is incomplete and requires you to finish it to make it work correctly. Refer to the links above under Evaluation, and back to the blogpost [here](https://medium.com/vickdata/a-simple-guide-to-scikit-learn-pipelines-4ac0d974bdcf) for explanation and examples of how to use the ColumnTransformer.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn import svm

model = svm.SVC()

# Define your preprocessing steps here by adding the features you want to One Hot Encode. DO NOT INCLUDE feature13. It will break the model
transformers = [("cat", OneHotEncoder(), [])]

# Combine steps into a ColumnTransformer
ct = ColumnTransformer(transformers=transformers)

# Apply your model to feature array X and labels y
def apply_model(model, X, y):    
    # Wrap the model and steps into a Pipeline
    pipeline = Pipeline(steps=[('t', ct), ('m', model)])
    
    # Evaluate the model and store results
    return evaluate_model(X, y, pipeline)

# Apply your validation techniques and calculate metrics
def evaluate_model(X, y, pipeline):
    pass

In [ ]:
apply_model(model, X, y)

### Label perturbation
To evaluate the impact of faulty labels in a dataset, we will introduce some errors in the labels of our data.


#### Preparation
Examine the method below which alters a dataset by selecting a percentage of rows and swaps labels from a 0->1 and 1->0. `sample` and `loc` are both methods available to work with DataFrames. Be sure to refer to the pandas docs for further information if the below is unclear.

In [ ]:
"""Create a new copy where 25% of the labels have been flipped."""
def perturbate(frame) -> pd.DataFrame:
    copy = frame.copy()
    # Select 25% of the data, and grab the index
    perturbers = copy.sample(frac=0.25, random_state=42).index
    # Using the indices, flip the labels
    flipped = copy.loc[perturbers, "target"].map({1: 0, 0: 1})
    # Assign those flipped labels back to their respective rows, using the indices
    copy.loc[perturbers, "target"] = flipped
    # return the DataFrame with the perturbed labels
    return copy

#### Analysis
Using the same process from the **Putting It All Together** subsection, re-run your SVC model with the new perturbed data. Again record the average accuracy of this new model, in the same manner as before.

In [ ]:
# Your code

Authors: Youri Arkesteijn, Tim van der Horst and Kevin Chong.

## Take It Further

In the lab above, we specified certain strategies or models at points throughout. From this point below, you are welcome to explore and play around with those we did not dedicate for instruction. Additionally, we have provided some links to get your exploration started!

An [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) is not the only type of classifier. There are many available, such as a [Decision Tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier), or [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression) and evaluate the performance in predicting the target variable. We encourage you to play around with these other models, and to explore others beyond these two available via Scikit-Learn.

What happens if you change the `frac=0.25` so a different value in the label perturbation?

## References

1. Dua, D. and Graff, C. (2019). UCI Machine Learning Repository [http://archive.ics.uci.edu/ml]. Irvine, CA: University of California, School of Information and Computer Science.

In [2]:
# Explore further here